## Burns' dialogism method

This notebook builds and verifies a Python port of the weighted log-odds "dialogism" method used by Patrick Burns in ["Measuring Dialogism in Latin Epic"](https://doi.org/10.1163/9789004750227_010), which we're aiming to replicate for Greek epic. Burns' method (itself based on Monroe, Colaresi & Quinn 2008, "Fightin' Words", as implemented in the R package `tidylo`) ranks features by a **weighted log-odds ratio with an uninformative Dirichlet prior**, then uses those rankings to build a composite per-token "dialogism" score.

Like `1 - Author and speech signal` and `2 - Rolling window`, this is a frozen record of how the method was built and verified, not a live view — the finished, packaged version lives in `ccc2026.dialogism`, which is what `4 - Compare speechiness methods` and `5 - Interactive` actually use.

### Import statements

In [1]:
import numpy as np
import pandas as pd
import ccc2026

### Load token data

In [2]:
# initialize package data if not done already
ccc2026.setup()

# load tokens using package helper method (cf. "manual" loading method in Notebooks 1 & 2)
tokens = ccc2026.tokens

### Label speech vs narrative

In [3]:
# Narratological groups (same definition used throughout the package)
nr_mask = tokens["speaker"].isna()
sp_mask = tokens["speaker"].notna() & tokens["speaker"].ne("Odysseus-Apologue")

### Weighted log-odds ratio

For each value of a feature column (lemma, POS tag, or morph tag), this computes a z-score: the log-odds ratio of speech vs. narrative, weighted by an uninformative Dirichlet prior. Following Monroe et al. (2008) and `tidylo`'s default ("uninformative") prior, the prior pseudo-count for a feature value is its own marginal (background) frequency across both narrative and speech combined.

Dividing the log-odds ratio (`delta`) by its own standard error (giving `z`) is what suppresses noise from rare features: a rare feature's ratio gets divided by a large standard error, pulling it back toward zero even if the raw ratio looks extreme.

In [4]:
def weighted_log_odds(col):
    '''Weighted log-odds ratio (speech vs. narrative) for every value of `col`,
    weighted by an uninformative Dirichlet prior (Monroe, Colaresi & Quinn 2008),
    following the same method used by Burns 2026 via the R package tidylo.
    '''

    # explode handles both scalar columns (lemma, pos) and list columns (morph)
    exploded = tokens[col].explode().dropna()
    is_nar = nr_mask.loc[exploded.index]
    is_spk = sp_mask.loc[exploded.index]

    y_nar = exploded[is_nar].value_counts()
    y_spk = exploded[is_spk].value_counts()
    counts = pd.DataFrame({"y_nar": y_nar, "y_spk": y_spk}).fillna(0)

    # uninformative Dirichlet prior: each feature's prior pseudo-count is its
    # own marginal (background) frequency across both groups combined
    alpha = counts["y_nar"] + counts["y_spk"]
    alpha0 = alpha.sum()
    n_nar = counts["y_nar"].sum()
    n_spk = counts["y_spk"].sum()

    omega_nar = (counts["y_nar"] + alpha) / (n_nar + alpha0 - counts["y_nar"] - alpha)
    omega_spk = (counts["y_spk"] + alpha) / (n_spk + alpha0 - counts["y_spk"] - alpha)

    delta = np.log(omega_spk) - np.log(omega_nar)
    variance = 1 / (counts["y_spk"] + alpha) + 1 / (counts["y_nar"] + alpha)
    z = delta / np.sqrt(variance)

    return pd.DataFrame({
        "count_nar": counts["y_nar"],
        "count_spk": counts["y_spk"],
        "delta": delta,
        "z": z,
    }).sort_values("z", ascending=False)

In [5]:
lemma_wlo = weighted_log_odds("lemma")

print("most speech-like lemmas:")
display(lemma_wlo.head(10))

print("most narrative-like lemmas:")
display(lemma_wlo.tail(10))

most speech-like lemmas:


,count_nar,count_spk,delta,z
lemma,,,,
ἐγώ,52.0,3526.0,0.788091,38.627828
σύ,65.0,3217.0,0.779649,36.645491
ἄν,254.0,1273.0,0.564612,18.628640
ἐμός,4.0,808.0,0.797400,18.575488
οὐ,876.0,2134.0,0.393402,18.509481
τίς,698.0,1814.0,0.410961,17.641107
εἰ,281.0,1209.0,0.533349,17.440897
νῦν,144.0,920.0,0.607805,16.654700
εἰμί,1172.0,2156.0,0.310355,15.429839


most narrative-like lemmas:


,count_nar,count_spk,delta,z
lemma,,,,
ῥʼ,354.0,55.0,-0.387606,-6.583996
πούς,610.0,160.0,-0.285033,-6.718465
φημί,847.0,259.0,-0.248723,-7.050104
πρόσφημι,230.0,1.0,-0.576828,-7.165961
ἕ,1325.0,494.0,-0.197663,-7.215683
ἀμφί,794.0,198.0,-0.296579,-7.925703
ὡς,2425.0,897.0,-0.200516,-9.890400
ἄρα,2134.0,415.0,-0.349343,-14.883579
ὁ,6713.0,2567.0,-0.194197,-16.020526


### Lexicons and composite dialogism score

Following Burns' "Methods" section: scale each ranked feature list to [0, 1] (closer to 1 = more speech-like) to produce a lexicon. Build one lexicon from lemmas (the lexical lexicon) and one combining POS and morph tags (the grammatical lexicon, matching how the package already treats POS+morph as the "grammatical" feature classes elsewhere).

Per token: the lexical score is the lemma's lexicon value (0 if out-of-vocabulary); the grammatical score is the mean of the lexicon values for the token's POS tag and all of its morph tags. The composite "dialogism" score is the average of the two.

In [6]:
def make_lexicon(wlo):
    '''Scale a weighted_log_odds table's z-scores to [0, 1] (1 = most speech-like)'''
    z = wlo["z"]
    return (z - z.min()) / (z.max() - z.min())

lex_lemma = make_lexicon(weighted_log_odds("lemma"))
lex_grammar = pd.concat([
    make_lexicon(weighted_log_odds("pos")),
    make_lexicon(weighted_log_odds("morph")),
])

# lexical score: lemma's lexicon value, 0 if out-of-vocabulary
lexical_score = tokens["lemma"].map(lex_lemma).fillna(0)

# grammatical score: mean of the POS tag + all morph tags' lexicon values
def grammar_score_for_row(pos, morph):
    vals = []
    if pd.notna(pos) and pos in lex_grammar.index:
        vals.append(lex_grammar[pos])
    if isinstance(morph, list):
        vals.extend(lex_grammar[m] for m in morph if m in lex_grammar.index)
    return np.mean(vals) if vals else np.nan

grammatical_score = tokens.apply(lambda row: grammar_score_for_row(row["pos"], row["morph"]), axis=1)

# composite dialogism score per token
dialogism_score = pd.concat(
    [lexical_score.rename("lex"), grammatical_score.rename("gram")], axis=1
).mean(axis=1, skipna=True)

dialogism_score.describe()

count    429774.000000
mean          0.392405
std           0.120088
min           0.086417
25%           0.322405
50%           0.361855
75%           0.417285
max           0.898369
dtype: float64

This matches `ccc2026.dialogism`, which is what the rest of the repo uses from here on:

- `weighted_log_odds` &rarr; `dialogism.weighted_log_odds`
- `make_lexicon` &rarr; `dialogism.make_lexicon`
- lexicon construction &rarr; `dialogism.build_lexicons()`
- composite score &rarr; `dialogism.token_dialogism_score(lexicons)`